# Travelers' Inn: exploratory data analysis

**Candidate:** TODO  
**Question:** At reservation confirmation, which bookings are most likely to be cancelled or become no-shows rather than complete as occupied stays?

This notebook is a presentation plan with runnable analysis cells. Replace every **TODO interpretation** with your own evidence-based explanation, remove prompts that no longer apply, run all cells, and save the outputs before submission.

## 1. Analysis contract and prediction cutoff

The prediction is made when a reservation is confirmed. The label combines cancellation and no-show because both leave a reserved room unoccupied. Fields created at outcome time or after confirmation are not valid predictors. Reviews are post-stay and therefore excluded from modeling, although their existence and join behaviour can be audited here.

**Before continuing, explain:** who uses the score, what low-harm action they can take, why price optimisation is out of scope, and what assumptions about feature availability need stakeholder confirmation.

In [ ]:
from pathlib import Path
import sqlite3

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src.data import clean_bookings, load_bookings

sns.set_theme(style="whitegrid", context="talk")
pd.set_option("display.max_columns", 60)
DB_PATH = Path("data/hotel.db")
assert DB_PATH.is_file(), "Place hotel.db at data/hotel.db; it is ignored by Git."

## 2. Load and understand the database

Purpose: verify the available tables, row counts, keys, types, and join grain before choosing any analysis. Never assume the PDF schema perfectly matches the physical database.

In [ ]:
with sqlite3.connect(f"file:{DB_PATH.resolve().as_posix()}?mode=ro", uri=True) as connection:
    tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table'", connection)
    reviews = pd.read_sql_query("SELECT * FROM reviews", connection)

raw = load_bookings(DB_PATH)
display(tables)
print(f"bookings: {len(raw):,} rows x {raw.shape[1]} columns")
print(f"reviews:  {len(reviews):,} rows x {reviews.shape[1]} columns")
display(raw.head(3))
display(pd.DataFrame({"dtype": raw.dtypes.astype(str), "non_null": raw.notna().sum(), "unique": raw.nunique(dropna=False)}))

**TODO interpretation:** Explain that SQLite stores all booking columns as text, identify the apparent row-index column, and state why key/grain verification comes before joining reviews. Check whether duplicated emails or many-to-many joins would multiply rows.

In [ ]:
key_audit = pd.Series({
    "booking rows": len(raw),
    "unique booking_id": raw["booking_id"].nunique(),
    "unique booking emails": raw["client_email"].nunique(),
    "review rows": len(reviews),
    "unique review emails": reviews["client_email"].nunique(),
    "rows after naive email join": len(raw.merge(reviews, on="client_email", how="inner")),
})
display(key_audit.to_frame("count"))

## 3. Data quality and deterministic cleaning

Purpose: quantify inconsistent encodings, blanks, category variants, duplicate IDs, and logically invalid values. Distinguish `NULL` from blank strings and placeholders. Cleaning rules must not use target rates or holdout statistics.

In [ ]:
blank_like = raw.apply(lambda s: s.astype("string").str.strip().isin(["", "-", "unknown", "NULL"]).sum())
display(blank_like.sort_values(ascending=False).head(15).to_frame("blank_or_placeholder"))
for column in ["is_canceled", "arrival_date_month", "meal", "market_segment", "deposit_type", "customer_type"]:
    print(f"\n{column}")
    display(raw[column].value_counts(dropna=False).head(20).to_frame("rows"))

clean, audit = clean_bookings(raw)
display(pd.Series(audit).to_frame("value"))
numeric_audit = clean[["lead_time", "adr", "adults", "children", "babies", "stays_in_week_nights", "stays_in_weekend_nights"]].describe(percentiles=[.01, .5, .99]).T
display(numeric_audit)

**TODO interpretation:** Quantify the important issues. Explain why duplicate booking IDs are removed, why negative lead time becomes missing, why robust scaling is preferable to deleting all extremes, and which apparent anomalies need domain confirmation (zero-night stays, zero guests, ADR extremes).

## 4. Target integrity, prevalence, and business framing

Purpose: prove that the binary flag agrees with reservation status and understand whether cancellation and no-show should be combined.

In [ ]:
target_cross = pd.crosstab(raw["is_canceled"], raw["reservation_status"], margins=True)
display(target_cross)
status_counts = raw["reservation_status"].value_counts().rename_axis("status").reset_index(name="rows")
status_counts["share"] = status_counts["rows"] / status_counts["rows"].sum()
display(status_counts)
ax = sns.barplot(data=status_counts, x="status", y="share", color="#2878B5")
ax.set(title="Reservation outcomes in the raw data", xlabel="Outcome", ylabel="Share of bookings")
ax.bar_label(ax.containers[0], labels=[f"{value:.1%}" for value in status_counts["share"]])
plt.show()

**TODO interpretation:** State exact counts/shares. Defend the combined outcome: it matches the shared empty-room decision, while no-show-only has too few positives for reliable modeling. Also explain that cancellation and no-show may later need separate interventions or a two-stage model.

## 5. Temporal and property patterns

Purpose: test whether prevalence changes by year, month, and hotel. This motivates a future-looking split and subgroup evaluation.

In [ ]:
clean["arrival_year"] = clean["arrival_date"].dt.year
clean["arrival_month_num"] = clean["arrival_date"].dt.month
year_hotel = clean.groupby(["arrival_year", "hotel"], as_index=False).agg(rows=("target", "size"), fallout_rate=("target", "mean"))
display(year_hotel)
ax = sns.lineplot(data=year_hotel, x="arrival_year", y="fallout_rate", hue="hotel", marker="o")
ax.set(title="Fallout rate changes over time and property", xlabel="Arrival year", ylabel="Cancellation/no-show rate")
plt.show()
monthly = clean.groupby(["arrival_year", "arrival_month_num"], as_index=False)["target"].mean()
pivot = monthly.pivot(index="arrival_year", columns="arrival_month_num", values="target").astype("float64")
sns.heatmap(pivot, annot=True, fmt=".2f", cmap="Blues")
plt.title("Monthly fallout rate by arrival year")
plt.xlabel("Arrival month")
plt.ylabel("Arrival year")
plt.show()

**TODO interpretation:** Describe stability/drift and sample sizes. Explain why random splitting would leak later operating conditions into training and why 2023/2024/2025 are used for train/validation/holdout.

## 6. Booking-time feature relationships

Purpose: find actionable associations without treating them as causal. Use rates together with group sizes and inspect numeric distributions rather than only correlations.

In [ ]:
categorical = ["hotel", "market_segment", "distribution_channel", "deposit_type", "customer_type", "promo_code"]
for column in categorical:
    summary = clean.groupby(column, dropna=False)["target"].agg(rows="size", fallout_rate="mean").query("rows >= 100").sort_values("fallout_rate", ascending=False)
    print(f"\n{column}")
    display(summary.head(15))

plot_frame = clean.assign(lead_time_band=pd.cut(clean["lead_time"], [-np.inf, 7, 30, 90, 180, np.inf]))
lead_summary = plot_frame.groupby("lead_time_band", observed=True)["target"].agg(rows="size", fallout_rate="mean").reset_index()
display(lead_summary)
sns.barplot(data=lead_summary, x="lead_time_band", y="fallout_rate", color="#F28E2B")
plt.title("Fallout rate by lead-time band")
plt.xlabel("Lead time (days)")
plt.ylabel("Cancellation/no-show rate")
plt.xticks(rotation=20)
plt.show()

**TODO interpretation:** For each selected chart/table, write (1) what is observed, (2) whether sample size supports it, (3) plausible business explanations, (4) why it may be confounded, and (5) whether it changes feature handling or evaluation. Do not infer causality from group rates.

## 7. Leakage audit

| Field | Decision | Reason at confirmation-time cutoff |
|---|---|---|
| `reservation_status`, `reservation_status_date` | Exclude | Direct outcome and outcome timestamp |
| `assigned_room_type` | Exclude | Actual assignment occurs at check-in |
| `booking_changes` | Exclude | Accumulates after booking creation; historical value cannot be reconstructed |
| `client_email`, `booking_id` | Exclude | PII/near-unique memorisation, not generalisable signal |
| reviews | Exclude | Written only after completed stays; structurally unavailable for fallout cases |
| `days_in_waiting_list` | Retain provisionally | Brief defines it as elapsed before confirmation |

**TODO:** Validate every retained field with a stakeholder/data-owner availability matrix. If scoring moves to seven days before arrival, rebuild historical point-in-time features rather than using final-row values.

## 8. Conclusions and pipeline decisions

Replace this section with a concise chain from evidence to decision:

1. **Problem and target:** exact population, outcome, cutoff, user, and action.
2. **Data quality:** which issues were corrected, flagged, retained, or excluded and why.
3. **Split:** why the temporal cohorts match deployment and what drift was observed.
4. **Features:** which booking-time patterns justify transformations; what was excluded for leakage.
5. **Models:** why logistic regression is the baseline and why a nonlinear alternative is worth testing.
6. **Metrics:** average precision, recall at contact capacity, ROC-AUC, Brier score, and a validation-only threshold.
7. **Limitations/next checks:** subgroup errors, calibration, uncertainty, causal pilot, missing business costs, and point-in-time availability.

Every conclusion should refer to a displayed statistic or visual above.